<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright (c) 2025 FlyDSL Project Contributors -->

# Numeric types

Every scalar you compute with in a FlyDSL kernel has a **DSL numeric type**.
These types are thin Python wrappers over MLIR scalar types: `fx.Int32` is `i32`,
`fx.Float32` is `f32`, `fx.BFloat16` is `bf16`, and so on. They carry the type
information FlyDSL needs to emit the right `arith` / `math` ops and to check casts.

This notebook covers the type families, how to construct and operate on values,
casts and promotion, and the difference between **compile-time** (`Constexpr`) and
**runtime** values.

In [ ]:
import torch

import flydsl.compiler as flyc
import flydsl.expr as fx
from wurlitzer import pipes


def show_gpu_output(launcher, *args, **kwargs):
    """Run a @flyc.jit launcher and echo its GPU printf output into the notebook.
    (Device printf isn't captured by Jupyter on its own; wurlitzer routes it here.)"""
    kwargs.setdefault("stream", torch.cuda.Stream())
    with pipes() as (out, _err):
        launcher(*args, **kwargs)
        torch.cuda.synchronize()
    print(out.read(), end="")

## 1. The type families

| Family | DSL types | MLIR |
|---|---|---|
| Signed int | `Int4`, `Int8`, `Int16`, `Int32`, `Int64` | `i4 … i64` |
| Unsigned int | `Uint8`, `Uint16`, `Uint32`, `Uint64` | `ui8 … ui64` |
| Float | `Float16`, `BFloat16`, `Float32`, `Float64` | `f16`, `bf16`, `f32`, `f64` |
| FP8 / FP6 / FP4 | `Float8E5M2`, `Float8E4M3FN`, …, `Float6E2M3FN`, `Float4E2M1FN` | OCP low-precision |
| Special | `Boolean` (`i1`), `Index` (loop/index type) | |

Each type exposes its bit `.width` (no GPU needed to read it):

In [ ]:
for t in [fx.Int8, fx.Int32, fx.Uint32, fx.Float16, fx.BFloat16,
          fx.Float32, fx.Float8E4M3FN, fx.Float4E2M1FN, fx.Boolean, fx.Index]:
    print(f"{t.__name__:<14} width = {t.width}")

## 2. Constructing and computing

A value is created by calling the type: `fx.Int32(7)`, `fx.Float32(3.5)`.

Arithmetic on these values **emits MLIR**, so it has to happen *inside a traced
kernel* (there is no MLIR context at the notebook top level). We surface the
results with `fx.printf`.

Operators map to the obvious MLIR ops: `+ - * // / %` → `arith.add/sub/mul/...`,
bitwise `& | ^ ~ << >>`, and comparisons `< <= > >= == !=` (which return a
`Boolean`). Two display quirks in `printf`: avoid a literal `%` in the format
string (the device `printf` consumes it), and a true `Boolean` prints as `-1`
(all bits of the `i1` set).

In [ ]:
@flyc.kernel
def numeric_demo():
    a = fx.Int32(7)
    b = fx.Int32(3)
    fx.printf("a+b={}  a-b={}  a*b={}  a//b={}  a-mod-b={}", a + b, a - b, a * b, a // b, a % b)
    fx.printf("a&b={}  a|b={}  a^b={}  a<<1={}", a & b, a | b, a ^ b, a << fx.Int32(1))
    fx.printf("a>b -> {} (true is -1)   a==b -> {}", a > b, a == b)


@flyc.jit
def run_numeric(stream: fx.Stream = fx.Stream(None)):
    numeric_demo().launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


show_gpu_output(run_numeric)

## 3. Casts and type promotion

Convert explicitly with `.to(TargetType)`. In a mixed-type expression FlyDSL also
**promotes** operands to a common type (wider wins; float beats int; on a tie,
unsigned beats signed). The `fx.math` module provides the usual float functions
(`sqrt`, `exp`, `exp2`, `log`, `sin`, `fma`, …).

In [ ]:
@flyc.kernel
def cast_demo():
    i = fx.Int32(7)
    f = i.to(fx.Float32)                # explicit int -> float
    g = fx.Float32(2.0)
    fx.printf("f/g={}  sqrt(f)={}  exp2(g)={}", f / g, fx.math.sqrt(f), fx.math.exp2(g))

    # mixed: Int32 + Float32 promotes to Float32
    promoted = fx.Int32(3) + g
    fx.printf("int(3) + float(2.0) = {}  (promoted to f32)", promoted)


@flyc.jit
def run_cast(stream: fx.Stream = fx.Stream(None)):
    cast_demo().launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


show_gpu_output(run_cast)

## 4. Compile-time (`Constexpr`) vs runtime values

A kernel parameter typed as `fx.Constexpr[T]` is a **Python value at trace time** —
it can size loops, pick code paths, and is baked into the compiled kernel (it is
part of the cache key, so a new value triggers a recompile). A parameter typed as a
numeric type (e.g. `fx.Int32`) is a **runtime IR value**, materialized in the kernel.

In [ ]:
@flyc.kernel
def constexpr_demo(scale: fx.Constexpr[int], x: fx.Int32):
    # `scale` is a real Python int here — we can use it in plain Python.
    label = "doubling" if scale == 2 else f"x{scale}"
    fx.printf("scale is a python int at trace time: " + label)
    fx.printf("x * scale = {}", x * scale)


@flyc.jit
def run_constexpr(x: fx.Int32, scale: fx.Constexpr[int], stream: fx.Stream = fx.Stream(None)):
    constexpr_demo(scale, x).launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


show_gpu_output(run_constexpr, fx.Int32(10), 4)

## 5. Low-precision types

AMD GPUs lean heavily on reduced precision, so the half- and sub-8-bit formats in
the table above (`BFloat16`, the `Float8*` / `Float6*` / `Float4*` families) are
first-class DSL types — you declare buffers and accumulators with them just like
`Float32`. Their narrow mantissas trade accuracy for speed and memory bandwidth.
The payoff shows up where data is *moved* and *multiplied* in low precision — the
copy atoms (next notebook) and the MMA atoms (the later layout/MMA series) — rather
than in scalar arithmetic, so we only introduce the type names here.

---
**Next:** [`02_struct`](02_struct.ipynb) — bundling these scalars into aggregate
value types with `@fx.struct`.